# 분포 확인하기

> 파이썬 7강 · 요약과 통계

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [분포 확인하기](https://mioon1402.github.io/timeseriesdata/python/p07-distribution.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

print('준비 완료')

## 1. 범주 세기 — value_counts

**7-1. 개수와 비율**

In [ ]:
import pandas as pd
df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])

print("요일별 일수")
print(df["weekday"].value_counts().to_string())

print("\n공휴일 비율 (%)")
print((df["is_holiday"].value_counts(normalize=True) * 100).round(1).to_string())

## 2. 숫자를 구간으로 묶기 — cut

**7-2. 기온대별로 묶어보기**

In [ ]:
df["기온대"] = pd.cut(
    df["avg_temp"],
    bins=[-20, 0, 10, 20, 30, 40],
    labels=["영하", "0~10도", "10~20도", "20~30도", "30도 이상"],
)

결과 = df.groupby("기온대", observed=True)["visitors"].agg(
    일수=("size"), 평균방문=("mean")
).round(1)
결과

## 3. 같은 개수로 나누기 — qcut

**7-3. 매출 5등급으로**

In [ ]:
df["매출등급"] = pd.qcut(df["sales"], 5, labels=["최하", "하", "중", "상", "최상"])

결과 = df.groupby("매출등급", observed=True)["sales"].agg(
    일수=("size"), 최소=("min"), 최대=("max")
).round(0)
결과

## 4. 이상치 기준 ① IQR 울타리

**7-4. IQR 울타리로 이상치 찾기**

In [ ]:
v = df["visitors"]
q1, q3 = v.quantile(0.25), v.quantile(0.75)
iqr = q3 - q1
아래, 위 = q1 - 1.5 * iqr, q3 + 1.5 * iqr

이상치 = df[(v < 아래) | (v > 위)]

print(f"Q1={q1:.0f}  Q3={q3:.0f}  IQR={iqr:.0f}")
print(f"울타리: {아래:.1f} ~ {위:.1f}")
print(f"이상치: {len(이상치)}일 ({len(이상치)/len(df)*100:.1f}%)")
print()
print("[너무 많았던 날]")
print(이상치.nlargest(3, "visitors")[["date", "weekday", "visitors"]].to_string(index=False))
print("\n[너무 적었던 날]")
print(이상치.nsmallest(3, "visitors")[["date", "weekday", "visitors"]].to_string(index=False))

## 5. 이상치 기준 ② 3시그마 — 그리고 왜 위험한가

**7-5. 두 기준 비교**

In [ ]:
m, s = v.mean(), v.std()
삼시그마 = df[(v < m - 3*s) | (v > m + 3*s)]

print(f"평균 {m:.1f}, 표준편차 {s:.1f}")
print(f"3σ 범위: {m - 3*s:.1f} ~ {m + 3*s:.1f}")
print()
print(f"3σ 기준으로 걸린 날 : {len(삼시그마):2d}일")
print(f"IQR 기준으로 걸린 날: {len(이상치):2d}일")
print()
print("3σ 가 놓친 것 중 하나:", 이상치.nsmallest(1, "visitors")[["date", "visitors"]].to_string(index=False))

**7-6. 가면 효과 직접 확인**

In [ ]:
깨끗 = v[(v >= 아래) & (v <= 위)]

print(f"전체 표준편차       {v.std():.1f}")
print(f"이상치 제외 표준편차 {깨끗.std():.1f}")
print(f"→ {(1 - 깨끗.std()/v.std())*100:.1f}% 부풀려져 있었습니다")

## 6. 이상치를 발견한 다음

**7-7. 이상치의 정체 확인하기**

In [ ]:
# 전체 열을 다 보면서 '왜 이런 값이 나왔는지' 단서를 찾는다
print(이상치.sort_values("visitors")[
    ["date", "weekday", "visitors", "sales", "avg_temp", "rain_mm", "is_holiday"]
].head(6).to_string(index=False))

**7-8. 지우지 말고 표시하기**

In [ ]:
# 지우는 대신 '이상치 여부' 열을 만들어두면 나중에 골라 쓸 수 있다
df["휴점"] = df["visitors"] == 0
df["이상치"] = (v < 아래) | (v > 위)

영업일 = df[~df["휴점"]]
평상시 = df[~df["이상치"]]

print(f"전체     {len(df):3d}일  평균 {df['visitors'].mean():.1f}명")
print(f"영업일만 {len(영업일):3d}일  평균 {영업일['visitors'].mean():.1f}명")
print(f"평상시만 {len(평상시):3d}일  평균 {평상시['visitors'].mean():.1f}명  "
      f"(SD {평상시['visitors'].std():.1f})")

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 매출(sales)에 대해 IQR 울타리를 구하고 이상치가 몇 일인지 세어보세요.


# 문제 2. 강수량(rain_mm)을 '없음 / 약함(0~10) / 보통(10~30) / 많음(30 이상)'
#        네 구간으로 나누고, 구간별 평균 방문객을 구해보세요.
#        힌트: pd.cut(df["rain_mm"], bins=[-0.1, 0, 10, 30, 100], labels=[...])


# 문제 3. 방문객을 qcut으로 3등분(하/중/상)하고, 등급별 평균 기온을 구해보세요.

**모범 답안**

In [ ]:
# 문제 1
s = df["sales"]
sq1, sq3 = s.quantile(0.25), s.quantile(0.75)
siqr = sq3 - sq1
s이상치 = df[(s < sq1 - 1.5*siqr) | (s > sq3 + 1.5*siqr)]
print(f"매출 이상치: {len(s이상치)}일")

# 문제 2
df["강수대"] = pd.cut(df["rain_mm"], bins=[-0.1, 0, 10, 30, 100],
                    labels=["없음", "약함", "보통", "많음"])
print()
print(df.groupby("강수대", observed=True)["visitors"]
        .agg(일수=("size"), 평균=("mean")).round(1).to_string())

# 문제 3
df["방문등급"] = pd.qcut(df["visitors"], 3, labels=["하", "중", "상"])
print()
(df.groupby("방문등급", observed=True)["avg_temp"]
   .agg(일수=("size"), 평균기온=("mean")).round(1))

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)